## Langfuse 프롬프트 관리 

### 1. Langfuse 계정 및 프로젝트 설정
1. [Langfuse Cloud](https://cloud.langfuse.com) 또는 Self-hosted 인스턴스에 가입
2. 프로젝트 생성 후 Settings에서 API 키 발급

### 2. 환경 변수 설정
`.env` 파일에 다음 내용을 추가하세요:
```
LANGFUSE_SECRET_KEY="sk-lf-..."
LANGFUSE_PUBLIC_KEY="pk-lf-..."
LANGFUSE_HOST="https://cloud.langfuse.com"
OPENAI_API_KEY="sk-..."
```

### 3. 필수 패키지 설치
```bash
uv add langfuse langchain-openai langchain-core python-dotenv
```

---

## 환경 설정 및 준비

### (1) Env 환경변수

In [6]:
from dotenv import load_dotenv
load_dotenv()

True

### (2) 기본 라이브러리

In [3]:
import os
from glob import glob
from pprint import pprint
import json
import warnings
warnings.filterwarnings("ignore")

### (3) Langfuse 콜백 핸들러 설정

In [12]:
from langfuse.langchain import CallbackHandler 

# LangChain 콜백 핸들러 생성
langfuse_handler = CallbackHandler()

### (4) Langfuse 클라이언트 설정

In [13]:
from langfuse import Langfuse

# Langfuse 클라이언트 초기화
langfuse = Langfuse()

# 연결 테스트
assert langfuse.auth_check()

---

## 프롬프트 관리 개요

Langfuse는 **프롬프트 CMS(Content Management System)** 기능을 제공

- **버전 관리**: 프롬프트의 모든 변경사항을 추적하고 롤백 가능
- **협업**: 팀원들과 함께 프롬프트를 편집하고 관리
- **배포 관리**: 라벨을 통해 코드 변경 없이 환경별 배포
- **성능 모니터링**: 프롬프트 버전별 성능 메트릭 비교
- **실시간 테스트**: 플레이그라운드에서 즉시 테스트 가능

- 라벨(Labels) 이해
    - `production`: 프로덕션 환경에서 사용 중인 안정화된 버전
    - `staging`: 테스트 환경에서 검증 중인 버전
    - `latest`: 가장 최근에 생성된 버전 (자동 부여)
    - 사용자 정의 라벨: 자유롭게 지정 가능 (`v2-stable`, `experiment-a` 등)

---

## 1. 프롬프트 생성

### 1.1 텍스트 프롬프트 생성

In [10]:
# 텍스트 프롬프트 생성
langfuse.create_prompt(
    name="movie-critic",  # 프롬프트 이름
    type="text",          
    prompt="{{criticLevel}} 영화 평론가로서, {{movie}}를 어떻게 생각하시나요?",
    labels=["production"],       # 프로덕션 레이블
    tags=["movie", "qa", "text"],    # 태그
    config={
        "model": "gpt-4.1-mini",      # 사용할 LLM 모델명
        "temperature": 0.7,            # 응답의 창의성 (0.0~2.0)
        "max_tokens": 500              # 최대 생성 토큰 수
    }
)

### 1.2 챗 프롬프트 생성

In [11]:
# 챗 프롬프트 생성
langfuse.create_prompt(
    name="movie-critic-chat",  # 프롬프트 이름
    type="chat",          
    prompt=[
        {
            "role": "system",
            "content": "당신은 {{criticLevel}} 영화 평론가입니다."
        },
        {
            "role": "user",
            "content": "영화 {{movie}}를 어떻게 생각하시나요?"
        }
    ],
    labels=["production"],       # 프로덕션 레이블
    tags=["movie", "qa", "chat"],    # 태그
    config={
        "model": "gpt-4.1-mini",      # 사용할 LLM 모델명
        "temperature": 0.7,            # 응답의 창의성 (0.0~2.0)
        "max_tokens": 500              # 최대 생성 토큰 수
    }
)

### 1.3 메시지 플레이스홀더가 있는 챗 프롬프트

In [12]:
# 메시지 플레이스홀더를 포함한 챗 프롬프트
langfuse.create_prompt(
    name="movie-critic-with-history",
    type="chat",
    prompt=[
        {
            "role": "system",
            "content": "당신은 {{criticLevel}} 영화 평론가입니다."
        },
        {
            "type": "placeholder",
            "name": "chat_history"  # 대화 히스토리 삽입 지점
        },
        {
            "role": "user",
            "content": "영화 {{movie}}에 대해 어떻게 생각하시나요?"
        }
    ],
    labels=["production"],
    tags=["movie", "qa", "chat", "history"],
    config={
        "model": "gpt-4.1-mini",      # 사용할 LLM 모델명
        "temperature": 0.7,            # 응답의 창의성 (0.0~2.0)
        "max_tokens": 500              # 최대 생성 토큰 수
    }
)

### **[실습 1]**
텍스트 기반 프롬프트와 chat 기반 프롬프트를 각각 구현하고, Langfuse UI에서 확인하세요.

**단계별 가이드:**
1. `langfuse.create_prompt()`를 사용하여 텍스트 프롬프트 생성
2. `type="text"`로 지정하고, 적절한 변수(`{{변수명}}`)를 포함한 프롬프트 작성
3. `type="chat"`으로 챗 프롬프트 생성 (system, user role 구분)
4. `config`에 모델명, temperature, max_tokens 설정
5. Langfuse UI의 Prompts 탭에서 생성된 프롬프트 확인

**힌트:** 
- 텍스트 프롬프트는 단순 문자열 형태
- 챗 프롬프트는 role과 content를 가진 딕셔너리 리스트 형태

In [13]:
# 텍스트 프롬프트 생성
# 여기에 코드를 작성하세요

In [14]:
# 챗 프롬프트 생성
# 여기에 코드를 작성하세요

---

## 2. 프롬프트 활용

### 2.1 기본 프롬프트 가져오기

In [15]:
# 프로덕션 버전 가져오기
prompt = langfuse.get_prompt("movie-critic")

# 프롬프트 정보 출력
print(f"모델: {prompt.config['model']}")
print(f"온도: {prompt.config['temperature']}")
print(f"라벨: {prompt.labels}")
print(f"태그: {prompt.tags}")
print(f"프롬프트: {prompt.prompt}")
print("-" * 100)

# 랭체인 호환 프롬프트 출력
print(prompt.get_langchain_prompt())

모델: gpt-4.1-mini
온도: 0.7
라벨: ['production', 'latest']
태그: ['movie', 'qa', 'text']
프롬프트: {{criticLevel}} 영화 평론가로서, {{movie}}를 어떻게 생각하시나요?
----------------------------------------------------------------------------------------------------
{criticLevel} 영화 평론가로서, {movie}를 어떻게 생각하시나요?


### 2.2 compile 메서드 사용

- compile 메서드로 변수 삽입

In [16]:
# compile 메서드로 변수 삽입
compiled_prompt = prompt.compile(criticLevel="전문가", movie="인셉션")
print(compiled_prompt)

전문가 영화 평론가로서, 인셉션를 어떻게 생각하시나요?


### 2.3 챗 프롬프트 가져오기 및 컴파일

In [17]:
# 챗 프롬프트 가져오기
chat_prompt = langfuse.get_prompt("movie-critic-chat", type="chat")

# 챗 프롬프트 정보 출력
print(f"모델: {chat_prompt.config['model']}")
print(f"온도: {chat_prompt.config['temperature']}")
print(f"라벨: {chat_prompt.labels}")
print(f"태그: {chat_prompt.tags}")
print(f"프롬프트: {chat_prompt.prompt}")
print("-" * 100)

# 랭체인 호환 프롬프트 출력
print(chat_prompt.get_langchain_prompt())

모델: gpt-4.1-mini
온도: 0.7
라벨: ['production', 'latest']
태그: ['movie', 'qa', 'chat']
프롬프트: [{'type': 'message', 'role': 'system', 'content': '당신은 {{criticLevel}} 영화 평론가입니다.'}, {'type': 'message', 'role': 'user', 'content': '영화 {{movie}}를 어떻게 생각하시나요?'}]
----------------------------------------------------------------------------------------------------
[('system', '당신은 {criticLevel} 영화 평론가입니다.'), ('user', '영화 {movie}를 어떻게 생각하시나요?')]


In [18]:
# 챗 프롬프트 컴파일
compiled_chat_prompt = chat_prompt.compile(criticLevel="전문가", movie="인셉션")
print(compiled_chat_prompt)

[{'role': 'system', 'content': '당신은 전문가 영화 평론가입니다.'}, {'role': 'user', 'content': '영화 인셉션를 어떻게 생각하시나요?'}]


### 2.4 메시지 플레이스홀더 활용

In [19]:
# 플레이스홀더가 있는 챗 프롬프트 가져오기
prompt_with_history = langfuse.get_prompt("movie-critic-with-history", type="chat")

# 대화 히스토리 정의
chat_history = [
    {"role": "user", "content": "안녕하세요!"},
    {"role": "assistant", "content": "안녕하세요! 영화에 대해 이야기해볼까요?"}
]

# 변수와 플레이스홀더를 모두 컴파일
compiled_with_history = prompt_with_history.compile(
            criticLevel="전문가",
            movie="인셉션", 
            chat_history=chat_history
        )

for message in compiled_with_history:
    print(message)
    print("-" * 20)

{'role': 'system', 'content': '당신은 전문가 영화 평론가입니다.'}
--------------------
{'role': 'user', 'content': '안녕하세요!'}
--------------------
{'role': 'assistant', 'content': '안녕하세요! 영화에 대해 이야기해볼까요?'}
--------------------
{'role': 'user', 'content': '영화 인셉션에 대해 어떻게 생각하시나요?'}
--------------------


### **[실습 2]**
"movie-critic-chat" 프롬프트를 Langfuse에서 가져와서 내용을 출력하고, compile 메서드를 사용해 변수에 적절한 값을 추가해보세요.

**단계별 가이드:**
1. `langfuse.get_prompt()`로 "movie-critic-chat" 프롬프트 가져오기
2. `type="chat"` 파라미터 지정
3. 프롬프트 정보 출력 (config, labels, prompt 등)
4. `compile()` 메서드로 `criticLevel`과 `movie` 변수에 값 할당
5. 컴파일된 메시지 출력

**힌트:**
- 챗 프롬프트는 메시지 리스트로 반환됨
- compile 결과는 for 문으로 순회하여 출력

In [20]:
# chat 프롬프트 가져오기 및 컴파일
# 여기에 코드를 작성하세요

---

## 3. 프롬프트 버전 관리

### 3.1 새로운 버전 생성

In [21]:
# 새로운 버전 생성 (같은 이름 사용)
langfuse.create_prompt(
    name="movie-critic",  # 같은 이름 사용
    type="text",          
    prompt="당신은 {{criticLevel}} 영화 평론가입니다.\n\n영화 {{movie}}에 대한 상세한 분석을 제공해주세요. 연출, 연기, 스토리, 시각적 효과를 포함하여 평가해주세요.",
    labels=["production"],       # 프로덕션 레이블
    tags=["movie", "qa", "text", "detailed"],    # 태그 업데이트
    config={
        "model": "gpt-4.1",  # 모델 업그레이드
        "temperature": 0.7,
        "max_tokens": 1000  # 토큰 수 증가
    }
)

### 3.2 특정 버전 가져오기

In [22]:
# 특정 버전 가져오기
prompt_v1 = langfuse.get_prompt("movie-critic", version=1)
prompt_v2 = langfuse.get_prompt("movie-critic", version=2)

# 버전별 비교
print(f"V1 프롬프트: {prompt_v1.prompt}")
print(f"V2 프롬프트: {prompt_v2.prompt}")
print(f"V1 모델: {prompt_v1.config['model']}")
print(f"V2 모델: {prompt_v2.config['model']}")

V1 프롬프트: {{criticLevel}} 영화 평론가로서, {{movie}}를 어떻게 생각하시나요?
V2 프롬프트: 당신은 {{criticLevel}} 영화 평론가입니다.

영화 {{movie}}에 대한 상세한 분석을 제공해주세요. 연출, 연기, 스토리, 시각적 효과를 포함하여 평가해주세요.
V1 모델: gpt-4.1-mini
V2 모델: gpt-4.1


### 3.3 라벨 관리

In [23]:
# 특정 라벨로 프롬프트 생성 (같은 이름을 사용하면 새로운 버전으로 생성됨)
langfuse.create_prompt(
    name="movie-critic-chat",
    type="chat",
    prompt=[
        {
            "role": "system",
            "content": "당신은 {{criticLevel}} 영화 평론가입니다. 상세하고 전문적인 분석을 제공해주세요."
        },
        {
            "role": "user",
            "content": "영화 {{movie}}에 대한 평론을 작성해주세요."
        }
    ],
    labels=["staging"],  # staging 환경용
    tags=["movie", "qa", "chat", "detailed"]
)

# 라벨별 프롬프트 가져오기
prompt_production = langfuse.get_prompt("movie-critic-chat", label="production")
prompt_staging = langfuse.get_prompt("movie-critic-chat", label="staging")
prompt_latest = langfuse.get_prompt("movie-critic-chat", label="latest")

In [24]:
# 라벨별 프롬프트 출력
print(f"Production 프롬프트: {prompt_production.prompt}")
print("-" * 100)
print(f"Staging 프롬프트: {prompt_staging.prompt}")
print("-" * 100)
print(f"Latest 프롬프트: {prompt_latest.prompt}")

Production 프롬프트: [{'type': 'message', 'role': 'system', 'content': '당신은 {{criticLevel}} 영화 평론가입니다.'}, {'type': 'message', 'role': 'user', 'content': '영화 {{movie}}를 어떻게 생각하시나요?'}]
----------------------------------------------------------------------------------------------------
Staging 프롬프트: [{'type': 'message', 'role': 'system', 'content': '당신은 {{criticLevel}} 영화 평론가입니다. 상세하고 전문적인 분석을 제공해주세요.'}, {'type': 'message', 'role': 'user', 'content': '영화 {{movie}}에 대한 평론을 작성해주세요.'}]
----------------------------------------------------------------------------------------------------
Latest 프롬프트: [{'type': 'message', 'role': 'system', 'content': '당신은 {{criticLevel}} 영화 평론가입니다. 상세하고 전문적인 분석을 제공해주세요.'}, {'type': 'message', 'role': 'user', 'content': '영화 {{movie}}에 대한 평론을 작성해주세요.'}]


### 3.4 라벨 업데이트

In [25]:
# 기존 프롬프트 버전의 라벨 업데이트
langfuse.update_prompt(
    name="movie-critic-chat",
    version=2,
    new_labels=["production", "v2-stable"]
)

Prompt_Chat(type='chat', prompt=[ChatMessage(role='system', content='당신은 {{criticLevel}} 영화 평론가입니다.', type=None), ChatMessage(role='user', content='<movie_title>\n{{movie_title}}\n</movie_title>\n\n<movie_content>\n{{movie_title}}\n</movie_content>', type=None)], name='movie-critic-chat', version=2, config={'model': 'gpt-4.1-mini', 'temperature': 0.7, 'max_tokens': 500}, labels=['production', 'v2-stable'], tags=['movie', 'qa', 'chat', 'detailed'], commit_message='xml tag 적용 및 프롬포트 수정', resolution_graph=None, id='5d1c0a50-b0a8-47c5-80c3-4f60dcc046a0', createdAt='2026-06-18T12:37:20.120Z', updatedAt='2026-06-19T11:30:17.338Z', projectId='cmqjg4h1x001oad0c1p2kzm9t', createdBy='cmqjg29ky007jad0e6lbu2zip', isActive=None)

### **[실습 3]**
"movie-critic-chat" 프롬프트를 수정하고, labels 속성은 "staging"으로 지정한 후, staging 버전을 가져와서 내용을 출력하세요.

**단계별 가이드:**
1. `langfuse.create_prompt()`로 같은 이름의 프롬프트 생성 (새 버전 생성)
2. `labels=["staging"]`으로 설정
3. 프롬프트 내용을 수정 (예: 더 상세한 지시사항 추가)
4. `langfuse.get_prompt(name="movie-critic-chat", label="staging")`으로 가져오기
5. 프롬프트 내용 출력 및 확인

**힌트:**
- 같은 이름으로 create하면 자동으로 새 버전 생성
- 라벨을 통해 환경별 프롬프트 관리 가능

In [26]:
# staging 라벨 생성
# 여기에 코드를 작성하세요
langfuse.create_prompt(
    name = "movie-critic-chat",
    type="chat",
    prompt=[
        {
            "role": "system",
            "content": "당신은 {{criticLevel}} 영화 평론가입니다. 영화에 대한 한줄평과 함께 상세한 분석을 작성해주세요."
        },
        {
            "role": "user",
            "content": "영화 {{movie}}에 대한 평론을 작성해주세요."
        }
    ],
    labels=["staging"],  # staging 환경용
    tags=["movie", "qa", "chat", "detailed"]
)


# staging 라벨 가져오기
# 여기에 코드를 작성하세요
#prompt_production = langfuse.get_prompt("movie-critic-chat", label="production")
prompt_staging = langfuse.get_prompt("movie-critic-chat", label="staging")
#prompt_latest = langfuse.get_prompt("movie-critic-chat", label="latest")

---

## 4. LangChain과의 통합

### 4.1 텍스트 프롬프트와 LangChain 통합

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# Langfuse 프롬프트를 LangChain과 통합
prompt = langfuse.get_prompt("movie-critic", label="production")

langchain_prompt = PromptTemplate.from_template(
    prompt.get_langchain_prompt(),
    metadata={"langfuse_prompt": prompt},  # Langfuse 자동 링크를 위한 메타데이터
)

# 모델 초기화 (프롬프트 설정 사용)
model = ChatOpenAI(
    model=prompt.config.get("model", "gpt-4.1-mini"),
    temperature=prompt.config.get("temperature", 0.7),
    max_tokens=prompt.config.get("max_tokens", 500)
)

# 체인 생성 및 실행
chain = langchain_prompt | model
response = chain.invoke(
    input={"criticLevel": "전문가", "movie": "인셉션"},
    config={"callbacks": [langfuse_handler]}  # Langfuse 트레이싱을 위한 콜백
)

print(response.content)

네, 영화 《인셉션》(Inception, 2010)은 크리스토퍼 놀란 감독의 대표작 중 하나로, 독창적인 스토리텔링과 뛰어난 연출, 배우들의 인상적인 연기, 혁신적인 시각적 효과로 전 세계적으로 큰 반향을 일으킨 작품입니다. 아래에 각 요소별로 상세하게 분석하겠습니다.

---

**1. 연출**

크리스토퍼 놀란 감독은 복잡한 구조의 스토리와 철학적 주제를 대중적으로 풀어내는 데 탁월한 재능을 보여줍니다. 《인셉션》은 ‘꿈 속의 꿈’이라는 다층적 내러티브 구조를 사용해, 현실과 꿈의 경계를 모호하게 만들며 관객 스스로 해석할 여지를 남깁니다. 각 층의 시간 흐름과 공간적 특성을 정교하게 배열해, 복잡하지만 혼란스럽지 않게 이야기를 이끌어 갑니다.

특히 꿈의 각 레벨마다 색채, 분위기, 연출 방식에 차이를 둬 관객이 현재 어느 레벨에 있는지 직관적으로 인식할 수 있게 하는 섬세함이 돋보입니다. 또한 영화의 마지막, 토템이 돌아가는 장면을 통해 결말을 명확히 제시하지 않고 여운을 남기는 방식 역시 놀란 감독 특유의 연출입니다.

---

**2. 연기**

주연 레오나르도 디카프리오(돔 코브 역)는 깊은 내면 연기와 감정 전달로 캐릭터의 상실, 죄책감, 집착을 설득력 있게 표현합니다. 특히 아내(말, 마리옹 꼬띠아르)와의 관계에서 드러나는 감정 연기는 영화의 심리적 깊이를 더합니다.

조셉 고든 레빗(아서 역), 엘렌 페이지(아리아드네 역), 톰 하디(임스 역), 켄 와타나베(사이토 역), 킬리언 머피(로버트 역) 등 조연들도 각자의 역할을 충실히 해내며 극에 활력을 불어넣습니다. 특히 아서와 임스의 티키타카는 액션과 유머를 동시에 선사합니다.

마리옹 꼬띠아르는 환상과 죄책감의 상징인 ‘말’ 역을 맡아, 불안정하고 미스터리한 존재감을 압도적으로 드러냅니다.

---

**3. 스토리**

《인셉션》의 중심 서사는 ‘타인의 꿈에 침투해 아이디어를 심는다’는 독특한 설정에서 출발합니다. 영화는 도둑질(익셉션)이 아니라 ‘인셉션(생각을 심는 것)’의 불가능성을 주

### 4.2 챗 프롬프트와 LangChain 통합

In [28]:
from langchain_core.prompts import ChatPromptTemplate

# 챗 프롬프트 통합
chat_prompt = langfuse.get_prompt("movie-critic-chat", label="production", type="chat")

langchain_chat_prompt = ChatPromptTemplate.from_messages(
    chat_prompt.get_langchain_prompt()
)
langchain_chat_prompt.metadata = {"langfuse_prompt": chat_prompt}

# 모델 초기화 (프롬프트 설정 사용)
model = ChatOpenAI(
    model=chat_prompt.config.get("model", "gpt-4.1-mini"),
    temperature=chat_prompt.config.get("temperature", 0.7),
    max_tokens=chat_prompt.config.get("max_tokens", 500)
)

# 체인 실행
chain = langchain_chat_prompt | model
response = chain.invoke(
    input={"criticLevel": "전문가", "movie_title": "인셉션"},
    config={"callbacks": [langfuse_handler]}
)

print(response.content)

영화 <인셉션>은 크리스토퍼 놀란 감독의 대표작으로, 복잡한 서사 구조와 독창적인 시각 효과가 돋보이는 작품입니다. 꿈속의 꿈이라는 다층적인 설정을 통해 현실과 환상의 경계를 모호하게 만들며, 관객에게 깊은 몰입감을 선사합니다. 레오나르도 디카프리오가 연기한 도미닉 코브는 기억과 죄책감에 시달리는 인물로, 그의 내면 갈등이 영화 전반에 걸쳐 긴장감을 유지시킵니다. 뛰어난 각본과 세밀한 연출, 그리고 한스 짐머의 강렬한 음악이 어우러져 시네마틱한 경험을 완성합니다. <인셉션>은 단순한 액션 영화가 아닌 철학적 질문을 던지는 작품으로, 반복 감상할수록 새로운 해석과 재미를 발견할 수 있는 걸작입니다.


### 4.3 플레이스홀더가 있는 프롬프트와 LangChain 통합

In [29]:
from langchain_core.prompts import MessagesPlaceholder

# 플레이스홀더가 있는 프롬프트 가져오기
prompt_with_history = langfuse.get_prompt("movie-critic-with-history", type="chat")

# LangChain 호환 프롬프트로 변환 (미해결 플레이스홀더는 MessagesPlaceholder로 변환)
langchain_prompt_with_placeholder = ChatPromptTemplate.from_messages(
    prompt_with_history.get_langchain_prompt()
)
langchain_prompt_with_placeholder.metadata = {"langfuse_prompt": prompt_with_history}

# 모델 초기화 (프롬프트 설정 사용)
model = ChatOpenAI(
    model=prompt_with_history.config.get("model", "gpt-4.1-mini"),
    temperature=prompt_with_history.config.get("temperature", 0.7),
    max_tokens=prompt_with_history.config.get("max_tokens", 500)
)

# chain 생성
chain = langchain_prompt_with_placeholder | model

# 실행 시 플레이스홀더 값 제공
chat_history = [
    {"role": "user", "content": "안녕하세요! 영화 예산에 대해서 이야기해볼까요?"},
    {"role": "assistant", "content": "안녕하세요! 영화 예산에 대해 어떻게 도와드릴까요?"}
]

response = chain.invoke({
    "criticLevel": "전문가", 
    "movie": "인셉션",
    "chat_history": chat_history
}, config={"callbacks": [langfuse_handler]})  # Langfuse 트레이싱을 위한 콜백

print(response.content)

Placeholders ['chat_history'] have not been resolved. Pass them as keyword arguments to compile().


영화 인셉션(Inception, 2010)은 크리스토퍼 놀란 감독의 대표작 중 하나로, 뛰어난 시나리오와 시각 효과, 그리고 탄탄한 연출이 돋보이는 작품입니다. 예산 측면에서 보면 약 1억 6천만 달러(약 1,600억 원) 정도가 투입되었는데, 이는 당시 기준으로 상당히 큰 규모의 제작비였습니다.

이 예산은 복잡한 시각 효과와 세트, 유명 배우들의 출연료, 그리고 촬영 기간 동안 다양한 장소에서의 촬영을 감안할 때 매우 효율적으로 사용되었다고 평가받습니다. 결과적으로 인셉션은 전 세계적으로 약 8억 3천만 달러 이상의 수익을 올리며, 투자 대비 높은 수익을 기록했죠.

예산이 크다고 해서 항상 좋은 영화가 나오는 것은 아니지만, 인셉션의 경우에는 그 예산이 작품의 완성도와 흥행에 큰 역할을 했다고 볼 수 있습니다. 혹시 인셉션의 특정 예산 항목이나 제작 과정에 대해 더 궁금한 점이 있으신가요?


### **[실습 4]**
앞에서 정의한 텍스트 기반 프롬프트를 가져와서 LangChain과 통합하여 트레이싱을 실행하고, Langfuse UI에서 결과를 확인하세요.

**단계별 가이드:**
1. `langfuse.get_prompt()`로 텍스트 프롬프트 가져오기
2. `PromptTemplate.from_template()`로 LangChain 프롬프트 생성
3. `metadata`에 Langfuse 프롬프트 링크 추가
4. `ChatOpenAI` 모델 초기화 (config에서 설정 값 가져오기)
5. 체인 생성 및 `invoke()` 실행 (callbacks에 langfuse_handler 추가)
6. Langfuse UI의 Traces 탭에서 실행 결과 확인

**힌트:**
- `prompt.config.get()` 메서드로 안전하게 설정 값 가져오기
- 콜백 핸들러를 추가해야 Langfuse에 트레이싱 기록됨


In [11]:
from langchain import chat_models
from langfuse.api import Traces
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
# 여기에 코드를 추가하세요.

# 1. `langfuse.get_prompt()`로 텍스트 프롬프트 가져오기
chat_prompt = langfuse.get_prompt("movie-critic", type="text")

# 2. `PromptTemplate.from_template()`로 LangChain 프롬프트 생성
# 3. `metadata`에 Langfuse 프롬프트 링크 추가
langchain_prompt = PromptTemplate.from_template(
    chat_prompt.get_langchain_prompt(),
    metadata={"langfuse_prompt": chat_prompt},
)

# 4. `ChatOpenAI` 모델 초기화 (config에서 설정 값 가져오기)
llm = ChatOpenAI(
    model=chat_prompt.config.get("model", "gpt-4.1-mini"),
    temperature=chat_prompt.config.get("temperature", 0.7),
    max_tokens=chat_prompt.config.get("max_tokens", 500)
)

# 5. 체인 생성 및 `invoke()` 실행 (callbacks에 langfuse_handler 추가)
chain = langchain_prompt | llm
response = chain.invoke(
    input={"criticLevel": "전문가", "movie": "인셉션"},
    config={"callbacks": [langfuse_handler]}
)

print(response)

# 6. Langfuse UI의 Traces 탭에서 실행 결과 확인



content='네, 영화 평론가의 시각에서 크리스토퍼 놀란 감독의 《인셉션》(Inception, 2010)에 대해 연출, 연기, 스토리, 시각적 효과 측면에서 상세하게 분석해드리겠습니다.\n\n---\n\n**1. 연출**\n\n크리스토퍼 놀란은 《인셉션》을 통해 ‘꿈’이라는 추상적이고 환상적인 개념을 치밀한 논리와 구조로 구현해냅니다. 영화는 다중의 꿈 속 꿈이라는 복잡한 서사를 선형적이면서도 혼란스럽지 않게 풀어내는 놀란 특유의 연출력이 돋보입니다. 놀란은 시간을 분할하고, 플롯을 병렬적으로 전개하며, 긴장감과 몰입감을 극대화합니다. 특히 마지막 40여 분간의 ‘킥’이 이루어지는 다층적 액션 시퀀스는 편집, 음악, 카메라 워크가 절묘하게 어우러져 명불허전의 긴장감을 선사합니다.\n\n**2. 연기**\n\n레오나르도 디카프리오는 주인공 ‘돔 코브’의 복잡한 내면을 섬세하게 표현합니다. 아내(마리옹 코티야르)와의 트라우마, 죄책감, 그리고 가족에 대한 그리움이 디카프리오의 깊이 있는 연기를 통해 설득력 있게 전달됩니다. 조셉 고든 레빗, 엘렌 페이지, 톰 하디, 켄 와타나베 등 조연들도 각자의 역할을 충실히 소화하며 앙상블을 이룹니다. 특히 마리옹 코티야르는 ‘말’ 역을 통해 사랑과 광기 사이의 아슬아슬한 경계를 인상적으로 그려냅니다.\n\n**3. 스토리**\n\n《인셉션》의 스토리는 단순한 ‘도둑질’에서 출발하지만, 인간 심리와 무의식을 탐구하는 메타포로 확장됩니다. 꿈의 세계를 설계하고, 타인의 무의식에 들어가 아이디어를 심는다는 독창적 설정은 관객의 상상력을 자극합니다. 각 등장인물의 사연과 동기가 유기적으로 얽혀있으며, ‘현실과 환상’의 경계에 대한 질문은 엔딩의 토템 신까지 이어져 철학적 여운을 남깁니다. 다만, 일부 관객에게는 복잡한 설정과 전문 용어가 진입 장벽으로 작용할 수 있습니다.\n\n**4. 시각적 효과**\n\n시각적 효과는 《인셉션》의 백미입니다. 꿈의 세계라는 콘셉트에 맞게 물리 법칙이 무시되는 공간 연출(파리 거리의 접힘, 중

AttributeError: 'LangchainCallbackHandler' object has no attribute 'flush'

### **[실습 5]**
앞에서 정의한 chat 기반 프롬프트를 가져와서 LangChain과 통합하여 트레이싱을 실행하고, Langfuse UI에서 결과를 확인하세요.

**단계별 가이드:**
1. `langfuse.get_prompt()`로 챗 프롬프트 가져오기 (`type="chat"` 지정)
2. `ChatPromptTemplate.from_messages()`로 LangChain 챗 프롬프트 생성
3. `metadata`에 Langfuse 프롬프트 링크 추가
4. `ChatOpenAI` 모델 초기화
5. 체인 생성 및 실행 (callbacks 포함)
6. Langfuse UI에서 트레이싱 결과 확인

**힌트:**
- 챗 프롬프트는 `ChatPromptTemplate.from_messages()` 사용
- 실습 4와 동일한 패턴으로 작성

In [31]:
# 여기에 코드를 추가하세요.